# Step 1: Setup prerequisites

In [1]:
import os
import sys
from pymongo import MongoClient

# utils에서 임포트하기 위해 부모 디렉토리를 경로에 추가합니다
sys.path.append(os.path.join(os.path.dirname(os.getcwd())))
from utils import set_env

In [2]:
# 본인의 MongoDB Atlas 클러스터를 사용하는 경우, 클러스터 연결 문자열을 여기에 입력하세요
MONGODB_URI = os.environ.get("MONGODB_URI")
# MongoDB Python 클라이언트 초기화
mongodb_client = MongoClient(MONGODB_URI)
# 서버 연결 확인
mongodb_client.admin.command("ping")

{'ok': 1.0,
 '$clusterTime': {'clusterTime': Timestamp(1773409902, 1),
  'signature': {'hash': b' ,\x93NR|Q\xfe\xdc\xc2\xcb\xf8\x02\xba~\xfao\xa3\xc5\xec',
   'keyId': 7616710838270820359}},
 'operationTime': Timestamp(1773409902, 1)}

In [3]:
# AI 모델 프록시 URL 설정
PROXY_ENDPOINT = os.environ.get("PROXY_ENDPOINT")

In [4]:
# 워크숍 강사로부터 제공받은 패스키를 설정하세요
PASSKEY = "replace-with-passkey"

In [5]:
# AI 모델 프록시에서 Voyage API 키를 가져와 환경변수로 설정합니다 -- 변경하지 마세요
set_env(["voyageai"], PASSKEY)

Successfully set VOYAGE_API_KEY environment variable.


# Step 2: Load the dataset

In [6]:
import json

In [7]:
with open("../data/mongodb_docs.json", "r") as data_file:
    json_data = data_file.read()

docs = json.loads(json_data)

In [8]:
# 데이터셋의 문서 수를 확인합니다
len(docs)

20

In [9]:
# 문서 구조를 파악하기 위해 첫 번째 문서를 미리 봅니다
docs[0]

{'updated': '2024-05-20T17:30:49.148Z',
 'metadata': {'contentType': None,
  'productName': 'MongoDB Atlas',
  'tags': ['atlas', 'docs'],
  'version': None},
 'action': 'created',
 'sourceName': 'snooty-cloud-docs',
 'body': '# View Database Access History\n\n- This feature is not available for `M0` free clusters, `M2`, and `M5` clusters. To learn more, see Atlas M0 (Free Cluster), M2, and M5 Limits.\n\n- This feature is not supported on Serverless instances at this time. To learn more, see Serverless Instance Limitations.\n\n## Overview\n\nAtlas parses the MongoDB database logs to collect a list of authentication requests made against your clusters through the following methods:\n\n- `mongosh`\n\n- Compass\n\n- Drivers\n\nAuthentication requests made with API Keys through the Atlas Administration API are not logged.\n\nAtlas logs the following information for each authentication request within the last 7 days:\n\n<table>\n<tr>\n<th id="Field">\nField\n\n</th>\n<th id="Description">\nD

# Step 3: Chunk and embed the data


In [10]:
# 이 셀 실행 시 경고가 표시될 수 있습니다 -- 무시해도 됩니다
from langchain.text_splitter import RecursiveCharacterTextSplitter
from typing import Dict, List
import voyageai
from tqdm import tqdm

In [11]:
# 텍스트 데이터에 사용할 구분자 목록
separators = ["\n\n", "\n", " ", "", "#", "##", "###"]

In [12]:
# LangChain의 `RecursiveCharacterTextSplitter`를 사용하여 위의 `separators` 목록 기준으로 텍스트를 분할합니다.
# 지정된 청크 크기에 도달할 때까지 재귀적으로 토큰을 병합합니다.
# 텍스트 데이터의 경우 일반적으로 하나의 청크에 1~2 단락(~200 토큰)을 유지합니다.
# 컨텍스트 임베딩의 경우 청크 오버랩을 0으로, 그렇지 않으면 청크 크기의 15~20%로 설정합니다.
# `model_name` 파라미터는 토크나이저로 사용할 인코더를 지정합니다(여기서는 GPT-4 인코더).
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    model_name="gpt-4", separators=separators, chunk_size=200, chunk_overlap=0
)

### 청킹(Chunking)이란?

긴 문서를 LLM이 처리할 수 있는 작은 조각으로 나누는 작업입니다.

**왜 청킹이 필요한가?**
- LLM은 한번에 처리할 수 있는 텍스트 길이(컨텍스트 윈도우)에 제한이 있습니다
- 문서 전체를 임베딩하면 의미가 희석됩니다 — 작은 단위일수록 검색 정확도가 높아집니다

**chunk_overlap을 0으로 설정하는 이유**

일반 청킹에서는 경계에서 문맥이 끊기는 것을 방지하기 위해 오버랩을 15~20%로 설정합니다.
하지만 이 노트북은 **컨텍스트 임베딩(Contextual Embedding)** 을 사용하기 때문에 오버랩이 불필요합니다.
LLM이 이미 각 청크에 전체 문서의 맥락을 주입하기 때문입니다. (아래 Step 3 설명 참고)

📚 https://reference.langchain.com/python/langchain-text-splitters/character/RecursiveCharacterTextSplitter/split_text

In [14]:
def get_chunks(doc: Dict, text_field: str) -> List[Dict]:
    """
    문서를 청크로 분할합니다.

    Args:
        doc (Dict): 청크를 생성할 원본 문서.
        text_field (str): 청크로 분할할 텍스트 필드.

    Returns:
        List[Dict]: 청크된 문서 목록.
    """
    # `doc`에서 청크할 필드를 추출합니다
    text = doc[text_field]
    # 위의 `text_splitter` 객체의 `split_text` 메서드로 `text`를 분할합니다
    chunks = text_splitter.split_text(text)
    return chunks

In [15]:
# Voyage AI 클라이언트 초기화
vo = voyageai.Client()

📚 https://docs.voyageai.com/docs/contextualized-chunk-embeddings#approach-2-contextualized-chunk-embeddings

### 컨텍스트 임베딩(Contextual Embedding)이란?

Anthropic이 2024년 발표한 방식으로, 청크를 임베딩하기 전에 LLM이 전체 문서 맥락을 요약해서 청크에 붙여줍니다.

**일반 임베딩 vs 컨텍스트 임베딩**

```
[일반 임베딩]
청크: "매출이 15% 증가했다."
→ 임베딩 모델 → 벡터 저장

[컨텍스트 임베딩]
전체 문서 + 청크 → LLM → "이 내용은 2024년 Q3 실적 보고서의 영업 성과 섹션입니다."
최종 청크: "이 내용은 2024년 Q3 실적 보고서의 영업 성과 섹션입니다. 매출이 15% 증가했다."
→ 임베딩 모델 → 벡터 저장
```

**왜 더 나은가?**
- 청크 자체가 "어느 문서의 어느 부분"인지 정보를 담고 있어 검색 정확도가 높아집니다
- Anthropic 실험 결과: retrieval 실패율 **49% 감소**

**trade-off**: 청크 하나당 LLM 호출이 1번씩 필요하므로 비용과 시간이 더 소요됩니다.

In [16]:
def get_embeddings(content: List[str], input_type: str) -> List[float] | List[List[float]]:
    """
    각 청크에 대한 컨텍스트 임베딩을 가져옵니다.

    Args:
        content (List[str]): 청크된 텍스트 목록 또는 리스트로 감싼 사용자 쿼리
        input_type (str): 입력 유형, "document" 또는 "query"

    Returns:
        List[float] | List[List[float]]: 컨텍스트 임베딩
    """
    # Voyage AI API의 `contextualized_embed` 메서드로 각 청크의 컨텍스트 임베딩을 가져옵니다:
    # inputs: `content`를 다른 리스트로 감싼 것
    # model: `voyage-context-3`
    # input_type: `input_type`
    embds_obj = vo.contextualized_embed(inputs=[content], model="voyage-context-3", input_type=input_type)
    # `input_type`이 "document"인 경우, 각 청크마다 하나의 임베딩이 있는 단일 결과가 반환됩니다
    if input_type == "document":
        embeddings = [emb for r in embds_obj.results for emb in r.embeddings]
    # `input_type`이 "query"인 경우, 단일 임베딩이 있는 단일 결과가 반환됩니다
    if input_type == "query":
        embeddings = embds_obj.results[0].embeddings[0]
    return embeddings

In [17]:
embedded_docs = []
# Step 2의 `docs`를 순회합니다
for doc in tqdm(docs):
    # `get_chunks` 함수로 각 문서의 "body" 필드를 청크로 분할합니다
    chunks = get_chunks(doc, "body")
    # 모든 `chunks`를 `get_embeddings` 함수에 전달하여 각 청크의 컨텍스트 임베딩을 가져옵니다
    # RAG용 문서를 임베딩하므로 `input_type`을 "document"로 설정합니다
    chunk_embeddings = get_embeddings(chunks, "document")
    # 각 청크에 대해 원본 메타데이터를 포함한 새 문서를 만듭니다
    # `body`를 청크 내용으로 교체하고 `embedding` 필드를 추가합니다
    for chunk, embedding in zip(chunks, chunk_embeddings):
        # 원본 문서를 복사하여 새 문서를 만듭니다
        chunk_doc = doc.copy()
        # `chunk_doc`의 `body` 필드를 청크 내용으로 교체합니다
        chunk_doc["body"] = chunk
        # 이 청크의 임베딩을 담은 `embedding` 필드를 `chunk_doc`에 추가합니다
        chunk_doc["embedding"] = embedding
        # `chunk_doc`을 `embedded_docs`에 추가합니다
        embedded_docs.append(chunk_doc)

100%|██████████| 20/20 [00:10<00:00,  1.82it/s]


In [18]:
# `embedded_docs`의 길이가 위 Step 2의 `docs`보다 큰 것을 확인합니다
# 각 문서가 여러 청크로 분할되었기 때문입니다
len(embedded_docs)

101

In [19]:
# 청크된 문서 구조를 파악하기 위해 첫 번째 문서를 미리 봅니다
# 원본 문서와 유사하지만 `body` 필드에 더 작은 텍스트 청크가 포함되어 있습니다
# 각 문서에 `embedding` 필드가 추가된 것도 확인할 수 있습니다
embedded_docs[0]

{'updated': '2024-05-20T17:30:49.148Z',
 'metadata': {'contentType': None,
  'productName': 'MongoDB Atlas',
  'tags': ['atlas', 'docs'],
  'version': None},
 'action': 'created',
 'sourceName': 'snooty-cloud-docs',
 'body': '# View Database Access History\n\n- This feature is not available for `M0` free clusters, `M2`, and `M5` clusters. To learn more, see Atlas M0 (Free Cluster), M2, and M5 Limits.\n\n- This feature is not supported on Serverless instances at this time. To learn more, see Serverless Instance Limitations.\n\n## Overview\n\nAtlas parses the MongoDB database logs to collect a list of authentication requests made against your clusters through the following methods:\n\n- `mongosh`\n\n- Compass\n\n- Drivers\n\nAuthentication requests made with API Keys through the Atlas Administration API are not logged.\n\nAtlas logs the following information for each authentication request within the last 7 days:\n\n<table>\n<tr>\n<th id="Field">\nField\n\n</th>\n<th id="Description">\nD

# Step 4: Ingest data into MongoDB


### **Do not change the values assigned to the variables below**

In [28]:
# 데이터베이스 이름 -- 필요시 변경하거나 그대로 두세요
DB_NAME = "mongodb_genai_devday_rag"
# 컬렉션 이름 -- 필요시 변경하거나 그대로 두세요
COLLECTION_NAME = "knowledge_base"
# 벡터 검색 인덱스 이름 -- 필요시 변경하거나 그대로 두세요
ATLAS_VECTOR_SEARCH_INDEX_NAME = "vector_index"

In [34]:
# `COLLECTION_NAME` 컬렉션에 연결합니다
collection = mongodb_client[DB_NAME][COLLECTION_NAME]

In [35]:
# 위에서 정의한 컬렉션의 모든 기존 레코드를 일괄 삭제합니다
collection.delete_many({})

DeleteResult({'n': 101, 'electionId': ObjectId('7fffffff0000000000000007'), 'opTime': {'ts': Timestamp(1773410929, 11), 't': 7}, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1773410929, 11), 'signature': {'hash': b'\x8fU,\xb4u\xae\x14`\x85\rC&\xc4\x91\xc6l\xfd\x92\xef\xcf', 'keyId': 7616710838270820359}}, 'operationTime': Timestamp(1773410929, 11)}, acknowledged=True)

📚 https://pymongo.readthedocs.io/en/stable/api/pymongo/collection.html#pymongo.collection.Collection.insert_many

In [36]:
# `embedded_docs`를 위에서 정의한 `collection`에 일괄 삽입합니다 -- 한 줄로 작성
collection.insert_many(embedded_docs)

print(f"Ingested {collection.count_documents({})} documents into the {COLLECTION_NAME} collection.")

Ingested 101 documents into the knowledge_base collection.


# Step 5: Create a vector search index

In [37]:
from utils import create_index, check_index_ready

In [38]:
# 벡터 인덱스 정의를 생성합니다:
# path: 임베딩 필드 경로
# numDimensions: 임베딩 차원 수 (사용하는 임베딩 모델에 따라 다름)
# similarity: 유사도 측정 방식. cosine, euclidean, dotProduct 중 하나
model = {
    "name": ATLAS_VECTOR_SEARCH_INDEX_NAME,
    "type": "vectorSearch",
    "definition": {
        "fields": [
            {
                "type": "vector",
                "path": "embedding",
                "numDimensions": 1024,
                "similarity": "cosine",
            }
        ]
    },
}

In [39]:
# `utils` 모듈의 `create_index` 함수로 `collection` 컬렉션에 위 정의로 벡터 검색 인덱스를 생성합니다
create_index(collection, ATLAS_VECTOR_SEARCH_INDEX_NAME, model)

Creating the vector_index index


In [40]:
# `utils` 모듈의 `check_index_ready` 함수로 인덱스가 생성되고 READY 상태인지 확인한 후 진행합니다
check_index_ready(collection, ATLAS_VECTOR_SEARCH_INDEX_NAME)

vector_index index status: PENDING
vector_index index status: READY
vector_index index definition: {'fields': [{'type': 'vector', 'path': 'embedding', 'numDimensions': 1024, 'similarity': 'cosine'}]}


# Step 6: Perform vector search on your data


### Define a vector search function

📚 https://www.mongodb.com/docs/atlas/atlas-vector-search/vector-search-stage/#ann-examples (Refer to the "ANN Basic Example")


In [41]:
# 벡터 검색으로 사용자 쿼리에 관련된 문서를 조회하는 함수를 정의합니다
def vector_search(user_query: str) -> List[Dict]:
    """
    벡터 검색으로 사용자 쿼리에 관련된 문서를 조회합니다.

    Args:
    user_query (str): 사용자 쿼리 문자열.

    Returns:
    list: 매칭된 문서 목록.
    """

    # Step 3에서 정의한 `get_embeddings` 함수로 `user_query`를 임베딩합니다
    # 참고: 함수가 문자열 목록을 기대하므로 user_query를 리스트로 감쌉니다
    # 쿼리를 임베딩하므로 `input_type`을 "query"로 설정합니다
    query_embedding = get_embeddings([user_query], "query")

    # $vectorSearch 스테이지와 $project 스테이지로 구성된 집계 파이프라인을 정의합니다
    # 후보 수는 150개로 설정하고 상위 5개만 반환합니다
    # $project 스테이지에서 `_id`는 제외하고 `body`, `metadata.productName`, `metadata.contentType`, `updated`, `vectorSearchScore`를 포함합니다
    # 참고: `index`, `queryVector`, `path` 필드에는 앞서 정의한 변수를 사용합니다
    pipeline = [
        {
            "$vectorSearch": {
                "index": ATLAS_VECTOR_SEARCH_INDEX_NAME,
                "queryVector": query_embedding,
                "path": "embedding",
                "numCandidates": 150,
                "limit": 5
            }
        },
        {
            "$project": {
                "_id": 0,
                "body": 1,
                "metadata.productName": 1, 
                "metadata.contentType": 1,
                "updated": 1,
                "score": {"$meta": "vectorSearchScore"}
            }
        }
    ]

    # 집계 `pipeline`을 실행하고 결과를 `results`에 저장합니다
    results = collection.aggregate(pipeline)
    return list(results)

### Run vector search queries


In [42]:
vector_search("What are some best practices for data backups in MongoDB?")

[{'updated': '2024-05-20T17:31:07.735Z',
  'metadata': {'contentType': None, 'productName': 'MongoDB Server'},
  'body': '# Backup and Restore Sharded Clusters\n\nThe following tutorials describe backup and restoration for sharded clusters:\n\nTo use `mongodump` and `mongorestore` as a backup strategy for sharded clusters, you must stop the sharded cluster balancer and use the `fsync` command or the `db.fsyncLock()` method on `mongos` to block writes on the cluster during backups.\n\nSharded clusters can also use one of the following coordinated backup and restore processes, which maintain the atomicity guarantees of transactions across shards:\n\n- MongoDB Atlas\n\n- MongoDB Cloud Manager\n\n- MongoDB Ops Manager\n\nUse file system snapshots back up each component in the sharded cluster individually. The procedure involves stopping the cluster balancer. If your system configuration allows file system backups, this might be more efficient than using MongoDB tools.\n\nCreate backups usi

In [43]:
vector_search("How to resolve alerts in MongoDB?")

[{'updated': '2024-05-20T17:30:49.148Z',
  'metadata': {'contentType': None, 'productName': 'MongoDB Atlas'},
  'body': '</td>\n<td headers="Description">\nPercentage of used disk space on a partition reaches a specified threshold.\n\n</td>\n</tr>\n<tr>\n<td headers="Alert%20Type">\nQuery Targeting Alerts\n\n</td>\n<td headers="Description">\nIndicates inefficient queries.\n\nThe change streams cursors that the Atlas Search process (`mongot`) uses to keep Atlas Search indexes updated can contribute to the query targeting ratio and trigger query targeting alerts if the ratio is high.\n\n</td>\n</tr>\n<tr>\n<td headers="Alert%20Type">\nReplica Set Has No Primary\n\n</td>\n<td headers="Description">\nNo primary is detected in replica set.\n\n</td>\n</tr>\n<tr>\n<td headers="Alert%20Type">\nReplication Oplog Alerts\n\n</td>\n<td headers="Description">\nAmount of oplog data generated on a primary cluster member is larger than the cluster\'s configured oplog size.',
  'score': 0.752524971961

# 🦹‍♀️ Combine pre-filtering with vector search

### Filter for documents where the product name is `MongoDB Atlas`

📚 https://www.mongodb.com/docs/atlas/atlas-vector-search/vector-search-type/#about-the-filter-type

In [ ]:
# Step 6의 벡터 검색 인덱스 `model`을 수정하여 `metadata.productName` 필드를 `filter` 필드로 추가합니다
model = {
    "name": ATLAS_VECTOR_SEARCH_INDEX_NAME,
    "type": "vectorSearch",
    "definition": {
        "fields": [
            {
                "type": "vector",
                "path": "embedding",
                "numDimensions": 1024,
                "similarity": "cosine"
            },
            {"type": "filter", "path": "metadata.productName"}
        ]
    }
}

In [ ]:
# `utils` 모듈의 `create_index` 함수로 수정된 모델로 벡터 검색 인덱스를 재생성합니다
create_index(collection, ATLAS_VECTOR_SEARCH_INDEX_NAME, model)

In [ ]:
# `utils` 모듈의 `check_index_ready` 함수로 인덱스에 올바른 필터 필드가 있고 READY 상태인지 확인한 후 진행합니다
check_index_ready(collection, ATLAS_VECTOR_SEARCH_INDEX_NAME)

In [ ]:
# 사용자 쿼리를 임베딩합니다
query_embedding = get_embeddings(
    ["What are some best practices for data backups in MongoDB?"], "query"
)

📚 https://www.mongodb.com/docs/atlas/atlas-vector-search/vector-search-stage/#ann-examples (Refer to the "ANN Filter Example")

In [ ]:
# Step 6에서 정의한 집계 파이프라인을 수정합니다:
# $vectorSearch 스테이지에 `metadata.productName` 필드 값이 "MongoDB Atlas"인 문서를 필터링하는 조건을 추가합니다
pipeline = [
    {
        "$vectorSearch": {
            "index": ATLAS_VECTOR_SEARCH_INDEX_NAME,
            "path": "embedding",
            "queryVector": query_embedding,
            "numCandidates": 150,
            "limit": 5,
            "filter": {"metadata.productName": "MongoDB Atlas"}
        }
    },
    {
        "$project": {
            "_id": 0,
            "body": 1,
            "metadata.productName": 1, 
            "metadata.contentType": 1,
            "updated": 1,
            "score": {"$meta": "vectorSearchScore"}
        }
    }
]

In [ ]:
# 집계 파이프라인을 실행하고 결과를 확인합니다
results = collection.aggregate(pipeline)
list(results)

### Filter on documents which have been updated on or after `2024-05-19` and where the content type is `Tutorial`

In [ ]:
# Step 6의 벡터 검색 인덱스 `model`을 수정하여 `metadata.contentType`과 `updated`를 `filter` 필드로 추가합니다
model = {
    "name": ATLAS_VECTOR_SEARCH_INDEX_NAME,
    "type": "vectorSearch",
    "definition": {
        "fields": [
            {
                "type": "vector",
                "path": "embedding",
                "numDimensions": 1024,
                "similarity": "cosine"
            },
            {"type": "filter", "path": "metadata.contentType"},
            {"type": "filter", "path": "updated"}
        ]
    }
}

In [ ]:
# `utils` 모듈의 `create_index` 함수로 수정된 모델로 벡터 검색 인덱스를 재생성합니다
create_index(collection, ATLAS_VECTOR_SEARCH_INDEX_NAME, model)

In [ ]:
# `utils` 모듈의 `check_index_ready` 함수로 인덱스에 올바른 필터 필드가 있고 READY 상태인지 확인한 후 진행합니다
check_index_ready(collection, ATLAS_VECTOR_SEARCH_INDEX_NAME)

In [ ]:
# 사용자 쿼리를 임베딩합니다
query_embedding = get_embeddings(
    ["What are some best practices for data backups in MongoDB?"], "query"
)

In [ ]:
# Step 6에서 정의한 집계 파이프라인을 수정합니다:
# `metadata.contentType` 필드가 "Tutorial"이고 `updated` 필드가 "2024-05-19" 이상인 문서를 필터링하는 조건을 추가합니다
# 힌트: >= 확인에는 $gte 연산자를, 조건 결합에는 $and 연산자를 사용합니다
pipeline = [
    {
        "$vectorSearch": {
            "index": ATLAS_VECTOR_SEARCH_INDEX_NAME,
            "path": "embedding",
            "queryVector": query_embedding,
            "numCandidates": 150,
            "limit": 5,
            "filter": {
                "$and": [
                    {"metadata.contentType": "Tutorial"},
                    {"updated": {"$gte": "2024-05-19"}}
                ]
            }
        }
    },
    {
        "$project": {
            "_id": 0,
            "body": 1,
            "metadata.productName": 1, 
            "metadata.contentType": 1,
            "updated": 1,
            "score": {"$meta": "vectorSearchScore"}
        }
    }
]

### RAG(Retrieval-Augmented Generation)란?

LLM이 학습 데이터에 없는 최신 정보나 특정 문서를 답변에 활용할 수 있도록, **검색(Retrieval)** 결과를 프롬프트에 함께 넣어주는 방법입니다.

```
사용자 질문
    ↓
[Retrieval] MongoDB 벡터 검색으로 관련 문서 청크 5개 조회
    ↓
[Augmentation] 프롬프트 = "다음 컨텍스트만 참고해서 답해줘:\n{문서 내용}\n\n질문: {사용자 질문}"
    ↓
[Generation] LLM이 컨텍스트 기반으로 답변 생성
```

**왜 RAG를 쓰는가?**
- LLM은 학습 시점 이후의 정보를 모릅니다
- 사내 문서, 최신 데이터 등 LLM이 학습하지 않은 정보를 활용할 수 있습니다
- 환각(Hallucination)을 줄일 수 있습니다 — "컨텍스트에 없으면 모른다고 답해"라고 지시 가능

In [ ]:
# 집계 파이프라인을 실행하고 결과를 확인합니다
results = collection.aggregate(pipeline)
list(results)

# Step 7: Build the RAG application


In [ ]:
import requests

### Define a function to create the chat prompt

In [ ]:
# RAG 애플리케이션의 사용자 프롬프트를 생성하는 함수를 정의합니다
def create_prompt(user_query: str) -> str:
    """
    사용자 쿼리와 검색된 컨텍스트를 포함하는 채팅 프롬프트를 생성합니다.

    Args:
        user_query (str): 사용자 쿼리 문자열.

    Returns:
        str: 채팅 프롬프트 문자열.
    """
    # Step 6에서 정의한 `vector_search` 함수로 `user_query`에 가장 관련성 높은 문서를 검색합니다
    context = vector_search(user_query)
    # 검색된 문서를 하나의 문자열로 합칩니다. 각 문서는 두 개의 줄바꿈("\n\n")으로 구분됩니다
    context = "\n\n".join([doc.get('body') for doc in context])
    # 질문과 관련 컨텍스트로 구성된 프롬프트
    prompt = f"Answer the question based only on the following context. If the context is empty, say I DON'T KNOW\n\nContext:\n{context}\n\nQuestion:{user_query}"
    return prompt

### Define a function to answer user queries

In [ ]:
# 사용자 쿼리에 답변하는 함수를 정의합니다
def generate_answer(user_query: str) -> None:
    """
    사용자 쿼리에 대한 답변을 생성합니다.

    Args:
        user_query (str): 사용자 쿼리 문자열.
    """
    # 위의 `create_prompt` 함수로 채팅 프롬프트를 생성합니다
    prompt = create_prompt(user_query)
    # LLM에 전달할 메시지를 [{"role": <role_value>, "content": <content_value>}] 형식으로 구성합니다
    # 사용자 메시지의 role 값은 "user"여야 합니다
    # 위에서 생성한 `prompt`로 채팅 메시지의 `content` 필드를 채웁니다
    messages = [{"role": "user", "content": prompt}]
    # AI 모델 프록시에 채팅 메시지를 전송하고 LLM 응답을 받습니다
    response = requests.post(url=PROXY_ENDPOINT, json={"task": "completion", "data": {"messages": messages}})
    if response.status_code == 200:
        # 최종 답변을 출력합니다
        print(response.json()["content"][0]["text"])
    else:
        # 오류 메시지를 출력합니다
        print(response.json()["error"])

### Query the RAG application


### Re-ranking(재순위)이란?

벡터 검색으로 가져온 문서를 **더 정교한 모델로 다시 정렬**하는 단계입니다.

```
벡터 검색 → 후보 문서 5개 (유사도 순)
    ↓
Re-ranker 모델 (쿼리 + 각 문서를 함께 분석)
    ↓
실제 관련성 순으로 재정렬된 문서 5개
```

**왜 필요한가?**

벡터 검색은 임베딩 벡터 간의 거리를 비교하므로 빠르지만, 쿼리와 문서를 **독립적으로** 임베딩합니다.
Re-ranker는 쿼리와 문서를 **함께** 보기 때문에 더 정확하지만 느립니다.

보통 다음 패턴으로 사용합니다:
1. 벡터 검색으로 후보 **50~100개** 빠르게 조회
2. Re-ranker로 상위 **5개** 정밀하게 선별
3. 선별된 5개를 LLM 프롬프트에 사용

In [ ]:
generate_answer("What are some best practices for data backups in MongoDB?")

In [ ]:
# 이 단계에서 LLM은 대화 기록을 기억하지 못하는 것을 확인합니다
generate_answer("What did I just ask you?")

# 🦹‍♀️ Re-rank retrieved results


### 메모리(Memory)란?

기본 LLM은 **대화 기록을 기억하지 못합니다**. 매 요청이 독립적이기 때문입니다.

```
[메모리 없음]
사용자: "MongoDB 백업 방법은?"  → LLM: "..."
사용자: "방금 뭐 물어봤지?"    → LLM: "모릅니다" ← 이전 대화를 모름

[메모리 있음]
사용자: "MongoDB 백업 방법은?"  → 저장 → MongoDB chat_history 컬렉션
사용자: "방금 뭐 물어봤지?"    → 이전 대화 조회 → LLM 프롬프트에 포함 → LLM: "백업 방법을 물어보셨습니다"
```

**이 노트북의 구현 방식**

- `store_chat_message()`: 사용자 메시지와 LLM 응답을 MongoDB에 저장
- `retrieve_session_history()`: `session_id`로 이전 대화 기록 조회
- `session_id`: 대화 세션 구분자 — 같은 ID면 같은 대화 흐름으로 인식

**왜 MongoDB에 저장하는가?**
- 서버를 재시작해도 대화 기록이 유지됩니다
- `session_id`로 여러 사용자의 대화를 독립적으로 관리할 수 있습니다

📚 https://docs.voyageai.com/docs/reranker#python-api (See Example)

In [ ]:
# 다음 함수에 재순위 단계를 추가합니다
def create_prompt(user_query: str) -> str:
    """
    사용자 쿼리와 검색된 컨텍스트를 포함하는 채팅 프롬프트를 생성합니다.

    Args:
        user_query (str): 사용자 쿼리 문자열.

    Returns:
        str: 채팅 프롬프트 문자열.
    """
    # Step 6에서 정의한 `vector_search` 함수로 `user_query`에 가장 관련성 높은 문서를 검색합니다
    context = vector_search(user_query)
    # `context`의 각 문서에서 "body" 필드를 추출합니다
    documents = [d.get("body") for d in context]
    # Voyage AI API의 `rerank` 메서드로 `documents`를 재순위합니다:
    # model: "rerank-2.5"
    # top_k: 5
    reranked_documents = vo.rerank(user_query, documents, model="rerank-2.5", top_k=5)
    # 재순위된 문서를 하나의 문자열로 합칩니다. 각 문서는 두 개의 줄바꿈("\n\n")으로 구분됩니다
    context = "\n\n".join([d.document for d in reranked_documents.results])
    # 질문과 관련 컨텍스트로 구성된 프롬프트
    prompt = f"Answer the question based only on the following context. If the context is empty, say I DON'T KNOW\n\nContext:\n{context}\n\nQuestion:{user_query}"
    return prompt

In [ ]:
# 재순위가 생성된 답변에 미치는 영향을 확인합니다
# 5개 문서만 재순위하므로 이 예제에서는 차이가 없을 수 있습니다
# 실제로는 더 많은 문서를 재순위기에 전달하고 재순위 후 상위 몇 개만 가져옵니다
generate_answer("What are some best practices for data backups in MongoDB?")

# Step 8: Add memory to the RAG application


In [ ]:
from datetime import datetime

In [ ]:
history_collection = mongodb_client[DB_NAME]["chat_history"]

📚 https://pymongo.readthedocs.io/en/stable/api/pymongo/collection.html#pymongo.collection.Collection.create_index


In [ ]:
# `history_collection` 컬렉션의 `session_id` 키에 인덱스를 생성합니다
history_collection.create_index("session_id")

### Define a function to store chat messages in MongoDB

📚 https://pymongo.readthedocs.io/en/stable/api/pymongo/collection.html#pymongo.collection.Collection.insert_one

In [ ]:
def store_chat_message(session_id: str, role: str, content: str) -> None:
    """
    채팅 메시지를 MongoDB 컬렉션에 저장합니다.

    Args:
        session_id (str): 메시지의 세션 ID.
        role (str): 메시지의 역할. `system`, `user`, `assistant` 중 하나.
        content (str): 메시지 내용.
    """
    # `session_id`, `role`, `content`, `timestamp` 필드가 있는 메시지 객체를 생성합니다
    # `timestamp`는 현재 타임스탬프로 설정합니다
    message = {
        "session_id": session_id,
        "role": role,
        "content": content,
        "timestamp": datetime.now(),
    }
    # `message`를 `history_collection` 컬렉션에 삽입합니다
    history_collection.insert_one(message)

### Define a function to retrieve chat history from MongoDB

📚 https://pymongo.readthedocs.io/en/stable/api/pymongo/cursor.html#pymongo.cursor.Cursor.sort

In [ ]:
def retrieve_session_history(session_id: str) -> List:
    """
    특정 세션의 채팅 메시지 기록을 조회합니다.

    Args:
        session_id (str): 채팅 메시지 기록을 조회할 세션 ID.

    Returns:
        List: 채팅 메시지 목록.
    """
    # "session_id" 필드 값이 입력 `session_id`와 같은 문서를 `history_collection`에서 조회합니다
    # 결과를 `timestamp` 필드의 오름차순으로 정렬합니다
    cursor = history_collection.find({"session_id": session_id}).sort("timestamp", 1)

    if cursor:
        # 커서를 순회하며 각 항목에서 `role`과 `content` 필드를 추출합니다
        # 각 항목을 {"role": <role_value>, "content": <content_value>} 형식으로 구성합니다
        messages = [{"role": msg["role"], "content": msg["content"]} for msg in cursor]
    else:
        # 커서가 비어있으면 빈 목록을 반환합니다
        messages = []

    return messages

### Handle chat history in the `generate_answer` function

In [ ]:
def generate_answer(session_id: str, user_query: str) -> None:
    """
    대화 기록을 고려하여 사용자 쿼리에 대한 답변을 생성합니다.

    Args:
        session_id (str): 대화 기록을 조회할 세션 ID.
        user_query (str): 사용자 쿼리 문자열.
    """
    # 채팅 완성 모델에 전달할 메시지 목록을 초기화합니다
    messages = []

    # 사용자 쿼리와 관련된 문서를 검색하고 하나의 문자열로 변환합니다
    context = vector_search(user_query)
    context = "\n\n".join([d.get("body", "") for d in context])
    # 검색된 컨텍스트를 포함하는 시스템 프롬프트를 생성합니다
    system_message = {
        "role": "user",
        "content": f"Answer the question based only on the following context. If the context is empty, say I DON'T KNOW\n\nContext:\n{context}",
    }
    # `messages` 목록에 시스템 프롬프트를 추가합니다
    messages.append(system_message)

    # `retrieve_session_history` 함수로 세션 ID `session_id`의 메시지 기록을 MongoDB에서 조회합니다
    # 메시지 기록의 모든 메시지를 `messages` 목록에 추가합니다
    message_history = retrieve_session_history(session_id)
    messages.extend(message_history)

    # 사용자 쿼리를 {"role": <role_value>, "content": <content_value>} 형식으로 구성합니다
    # 사용자 메시지의 role 값은 "user"여야 합니다
    # 사용자 메시지를 `messages` 목록에 추가합니다
    user_message = {"role": "user", "content": user_query}
    messages.append(user_message)

    # AI 모델 프록시에 채팅 메시지를 전송하고 LLM 응답을 받습니다
    response = requests.post(url=PROXY_ENDPOINT, json={"task": "completion", "data": {"messages": messages}})

    # 응답에서 답변을 추출합니다
    answer = response.json()["content"][0]["text"]

    # `store_chat_message` 함수로 사용자 메시지와 생성된 답변을 메시지 기록 컬렉션에 저장합니다
    # 사용자 메시지의 role 값은 "user", 생성된 답변은 "assistant"입니다
    store_chat_message(session_id, "user", user_query)
    store_chat_message(session_id, "assistant", answer)

    print(answer)

In [ ]:
generate_answer(
    session_id="1",
    user_query="What are some best practices for data backups in MongoDB?",
)

In [ ]:
generate_answer(
    session_id="1",
    user_query="What did I just ask you?",
)